In [116]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, RepeatedKFold, LeaveOneOut, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

In [117]:
# Load Data ---------------------------------------------------------------
# Carregando o dataset California Housing (Regressão)
try:
    df = pd.read_csv("./03-Database/Workbook/WB7 - CV- Sacramento.csv")
except FileNotFoundError:
    print("ERRO: O arquivo 'WB7 - CV- Sacramento.csv' não foi encontrado.")
    print("Certifique-se de que o caminho está correto.")
    exit(1)

# Exemplo de como carregar uma base de dados em CSV (descomente e ajuste o caminho se quiser usar um CSV)
print(df.describe())

ERRO: O arquivo 'WB7 - CV- Sacramento.csv' não foi encontrado.
Certifique-se de que o caminho está correto.
             beds       baths         sqft          price    latitude  \
count  932.000000  932.000000   932.000000     932.000000  932.000000   
mean     3.275751    2.053112  1680.318670  246661.583691   38.592980   
std      0.887998    0.722599   726.266383  131126.862021    0.134159   
min      1.000000    1.000000   484.000000   30000.000000   38.241514   
25%      3.000000    2.000000  1166.750000  156000.000000   38.478436   
50%      3.000000    2.000000  1470.000000  220000.000000   38.619343   
75%      4.000000    2.000000  1953.500000  305000.000000   38.686063   
max      8.000000    5.000000  4878.000000  884790.000000   39.020808   

        longitude  
count  932.000000  
mean  -121.360288  
std      0.140178  
min   -121.551704  
25%   -121.449257  
50%   -121.382474  
75%   -121.308444  
max   -120.597599  


In [118]:
# Análise Exploratória de Dados -------------------------------------------
print("--- Análise Exploratória de Dados ---")
print("Primeiras Linhas (head):")
print(df.head())
print("\nResumo Estatístico (summary):")
print(df.describe(include='all'))

--- Análise Exploratória de Dados ---
Primeiras Linhas (head):
         city     zip  beds  baths  sqft  price   latitude   longitude  \
0  SACRAMENTO  z95838     2    1.0   836  59222  38.631913 -121.434879   
1  SACRAMENTO  z95823     3    1.0  1167  68212  38.478902 -121.431028   
2  SACRAMENTO  z95815     2    1.0   796  68880  38.618305 -121.443839   
3  SACRAMENTO  z95815     2    1.0   852  69307  38.616835 -121.439146   
4  SACRAMENTO  z95824     2    1.0   797  81900  38.519470 -121.435768   

   type_Condo  type_Multi_Family  type_Residential  
0       False              False              True  
1       False              False              True  
2       False              False              True  
3       False              False              True  
4       False              False              True  

Resumo Estatístico (summary):
              city     zip        beds       baths         sqft  \
count          932     932  932.000000  932.000000   932.000000   
unique   

In [119]:
print("\n--- Pré-processamento ---")

# 1. Tratar valores ausentes: Remover linhas com NAs
housing_data = df.dropna().copy()

# 2. Converter strings (variáveis 'character' em R) em categóricas (fatores em R)
for col in housing_data.columns:
    if housing_data[col].dtype == 'object':
        print(f"Convertendo coluna '{col}' para categórica.")
        housing_data[col] = housing_data[col].astype('category')


--- Pré-processamento ---
Convertendo coluna 'city' para categórica.
Convertendo coluna 'zip' para categórica.


In [120]:
print("\nResumo Estatístico (summary):")
print(df.describe(include='all'))


Resumo Estatístico (summary):
              city     zip        beds       baths         sqft  \
count          932     932  932.000000  932.000000   932.000000   
unique          37      68         NaN         NaN          NaN   
top     SACRAMENTO  z95823         NaN         NaN          NaN   
freq           438      61         NaN         NaN          NaN   
mean           NaN     NaN    3.275751    2.053112  1680.318670   
std            NaN     NaN    0.887998    0.722599   726.266383   
min            NaN     NaN    1.000000    1.000000   484.000000   
25%            NaN     NaN    3.000000    2.000000  1166.750000   
50%            NaN     NaN    3.000000    2.000000  1470.000000   
75%            NaN     NaN    4.000000    2.000000  1953.500000   
max            NaN     NaN    8.000000    5.000000  4878.000000   

                price    latitude   longitude type_Condo type_Multi_Family  \
count      932.000000  932.000000  932.000000        932               932   
unique  

In [121]:
# Variáveis dummy para a variável categórica 'type'
# Presumindo que 'type' é a variável categórica relevante para a Regressão
df = pd.get_dummies(df, columns=['type'], drop_first=False)

df.head()

KeyError: "None of [Index(['type'], dtype='object')] are in the [columns]"

In [ ]:
X = df[['beds', 'baths', 'sqft'] + [col for col in df.columns if col.startswith('type_')]]
y = df[['price']]

print(X.head())
print(y.head())

   beds  baths  sqft  type_Condo  type_Multi_Family  type_Residential
0     2    1.0   836       False              False              True
1     3    1.0  1167       False              False              True
2     2    1.0   796       False              False              True
3     2    1.0   852       False              False              True
4     2    1.0   797       False              False              True
   price
0  59222
1  68212
2  68880
3  69307
4  81900


In [ ]:
# Ajustando o modelo de regressão linear com y e X
model = LinearRegression()
# model.set_params(fit_intercept=False)
model.fit(X, y)



,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [ ]:
# 2. Acessar e Apresentar os Coeficientes

print("--- Coeficientes do Modelo (scikit-learn) ---")

# a) Intercepto (Termo Constante)
intercepto = model.intercept_
print(f"Intercepto (Constante): {intercepto[0]:.4f}")

print("\nCoeficientes das Variáveis Independentes:")
# b) Coeficientes (Atributo .coef_)
coeficientes = model.coef_

# Criar um DataFrame para visualização clara (como uma tabela)
df_coefs = pd.DataFrame({
    'Variável': X.columns,
    'Coeficiente': coeficientes[0].round(4)
})

print(df_coefs)
print("---------------------------------------------")


--- Coeficientes do Modelo (scikit-learn) ---
Intercepto (Constante): 36816.9502

Coeficientes das Variáveis Independentes:
            Variável  Coeficiente
0               beds  -29788.0100
1              baths    8730.0188
2               sqft     156.1612
3         type_Condo   -3897.5975
4  type_Multi_Family  -25893.3969
5   type_Residential   29790.9944
---------------------------------------------


In [ ]:
X2 = X[['beds', 'baths', 'sqft']+ [col for col in X.columns if col.startswith('type_')]]
print(X2)
y2 = y['price']
X = sm.add_constant(X)  # Add a constant for the intercept
ols_mdl = sm.OLS(y2, X2).fit()

# The R `summary(ols.mdl)` is equivalent to the following
print("--- Model Summary (statsmodels) ---")
print(ols_mdl.summary())
print("-----------------------------------")


     beds  baths  sqft  type_Condo  type_Multi_Family  type_Residential
0       2    1.0   836       False              False              True
1       3    1.0  1167       False              False              True
2       2    1.0   796       False              False              True
3       2    1.0   852       False              False              True
4       2    1.0   797       False              False              True
..    ...    ...   ...         ...                ...               ...
927     4    3.0  2280       False              False              True
928     3    2.0  1477       False              False              True
929     3    2.0  1216       False              False              True
930     4    2.0  1685       False              False              True
931     3    2.0  1362       False              False              True

[932 rows x 6 columns]


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

In [ ]:

# --- 1. k-Fold Cross-Validation ---
print("\n--- 1. k-Fold Cross-Validation (k=10) ---")
k_fold = KFold(n_splits=10, shuffle=True, random_state=123)
model_kfold = LinearRegression()

scores_kfold = cross_val_score(model_kfold, X, y, cv=k_fold, scoring='neg_mean_squared_error')

# O caret retorna as métricas resumidas.
print(f"Média do MSE (10 Folds): {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão: {np.std(scores_kfold):.4f}")



--- 1. k-Fold Cross-Validation (k=10) ---
Média do MSE (10 Folds): -6830366387.8599
Desvio Padrão do MSE: 1953389241.4982


In [ ]:
# --- 2. Repeated k-Fold Cross-Validation ---
print("\n--- 2. Repeated k-Fold Cross-Validation (10 Folds, 3 Repetições) ---")
rep_kf = RepeatedKFold(n_splits=10, n_repeats=3, random_state=123)
model_repeated = LinearRegression()

scores_repeated = cross_val_score(model_repeated, X, y, cv=rep_kf, scoring='neg_mean_squared_error')

print(f"Média do MSE (30 Iterações): {np.mean(scores_repeated):.4f}")
print(f"Desvio Padrão: {np.std(scores_repeated):.4f}")



--- 2. Repeated k-Fold Cross-Validation (10 Folds, 3 Repetições) ---
Média do R² (30 Iterações): -6818629500.9214
Desvio Padrão do R²: 1862657736.0232


In [89]:
# --- 3. Leave-One-Out Cross-Validation (LOOCV) ---
print("\n--- 3. Leave-One-Out Cross-Validation (LOOCV) ---")
# LOOCV é computacionalmente intensivo para datasets grandes, mas é implementado aqui.
loocv = LeaveOneOut()
model_loocv = LinearRegression()

# Aviso: Este passo pode demorar em datasets grandes.
scores_loocv = cross_val_score(model_loocv, X, y, cv=loocv, scoring='neg_mean_squared_error')

print(f"Média do MSE (n Iterações): {np.mean(scores_loocv):.4f}")
print(f"Desvio Padrão: {np.std(scores_loocv):.4f}")



--- 3. Leave-One-Out Cross-Validation (LOOCV) ---
Média do MSE (n Iterações): -6823271765.6360
Desvio Padrão: 18450473594.3879
